# 02.02 · Tool calling without an agent

**Module 02 · Tools & Your First Agents** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 02.01 (tools). Lesson 01.02 (messages). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Bind** tools to a model with `bind_tools` and see exactly what goes over the wire (`tools`, `tool_choice`).
2. **Read** a tool-calling reply: `tool_calls`, `content_blocks`, `finish_reason`, `invalid_tool_calls`.
3. **Control** tool use with `tool_choice` (`auto`, `required`/`any`, a tool name) and know Groq's limits (no parallel calls on GPT-OSS).
4. **Execute** tool calls and return `ToolMessage`s that follow the message-ordering rules.
5. **Write the agent loop by hand**, with a stop condition and a step limit, and handle bad tool calls without crashing.
6. **Recognize** server-side tools (Groq's `browser_search`) and the OpenAI-compatible fallback (`ChatOpenAI` pointed at Groq).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Models → Tool calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling) | **Main page.** `bind_tools`, the tool execution loop, forcing tool calls, parallel calls, streaming tool calls |
| [Messages → Tool message](https://docs.langchain.com/oss/python/langchain/messages#tool-message) | `ToolMessage`, `tool_call_id`, `artifact`, `status` |
| [Groq: local tool calling](https://console.groq.com/docs/tool-use/local-tool-calling) | The same loop in the raw Groq SDK: request, `tool_calls`, `role: "tool"` replies |
| [Groq: tool use overview](https://console.groq.com/docs/tool-use/overview) | Supported models, parallel tool use, built-in vs local tools |
| [Groq: browser search](https://console.groq.com/docs/tool-use/built-in-tools/browser-search) | A **server-side** tool: Groq runs the search loop for you |
| [Groq: OpenAI compatibility](https://console.groq.com/docs/openai) | Base URL `https://api.groq.com/openai/v1` and unsupported fields |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/02-tools-and-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · `bind_tools`: what actually gets sent

`bind_tools(tools)` returns a **new Runnable** (a binding; in langchain-core 1.6 its class is `_ChatModelBinding`, a `RunnableBinding` subclass): the same model with extra request parameters attached. Nothing is sent until you call it. We can inspect the binding **offline**: building a `ChatGroq` object doesn't touch the network.

In [ ]:
# 🧪 Offline: bind two tools to a ChatGroq object and look at the attached request parameters.
from langchain_groq import ChatGroq
from mastery.tools import calculator, days_between

offline_groq = ChatGroq(model=settings.model_fast, api_key="gsk_offline_inspection_only")   # Never called.
bound = offline_groq.bind_tools([calculator, days_between])

print(type(bound).__name__)
print("request kwargs:", list(bound.kwargs))
for spec in bound.kwargs["tools"]:
    fn = spec["function"]
    print(f"- {fn['name']}: {fn['description'][:70]}... args={list(fn['parameters']['properties'])}")

**You should now see** `_ChatModelBinding`, `request kwargs: ['tools']`, and the two tools in OpenAI/Groq format. Every call through `bound` sends these schemas, so **tools cost input tokens on every request**, whether or not they are used. Ten verbose tools can add a thousand tokens to each step of an agent.

---
## Part 2 · Reading a tool-calling reply

When the model decides to use a tool, its `AIMessage` usually has **empty `content`** and a list of `tool_calls`. Each call has a `name`, parsed `args` (a dict), an `id` and `type="tool_call"`. The `id` matters: your answer must quote it.

We start with a scripted model (`mastery.testing.ScriptedChatModel`) so the shape is predictable, then do it live.

In [ ]:
# 🧪 Offline: a scripted model that "decides" to call days_between.
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from mastery.testing import ScriptedChatModel

scripted = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "days_between", "args": {"start": "2026-10-12", "end": "2026-12-25"},
                               "id": "call_abc123"}]),
])
reply = scripted.bind_tools([calculator, days_between]).invoke("How many days until Christmas 2026? Today is 2026-10-12.")

print("content       :", repr(reply.content))
print("tool_calls    :", reply.tool_calls)
print("content_blocks:", reply.content_blocks)

In [ ]:
# 💸 Live: the same question to the real model (1 short call).
QUESTION = "How many days are there from 2026-10-12 to 2026-12-25? Use a tool."

if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    llm_with_tools = llm.bind_tools([calculator, days_between])
    live_reply = llm_with_tools.invoke(QUESTION)
    print("tool_calls   :", live_reply.tool_calls)
    print("finish_reason:", live_reply.response_metadata.get("finish_reason"))
    print("invalid      :", live_reply.invalid_tool_calls)
    print("tokens       :", live_reply.usage_metadata)
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** one call to `days_between` with `{'start': '2026-10-12', 'end': '2026-12-25'}` and `finish_reason: tool_calls`. Look at `input_tokens`: the question is ~20 tokens; the rest is the tool schemas (and the model's built-in system prompt).

### What just happened?

- `finish_reason == "tool_calls"` means *"I stopped because I want a tool"*. It is **not** the final answer.
- The provider returned the arguments as a JSON **string**; LangChain parsed it into `args`. If the JSON was broken, the call would land in `invalid_tool_calls` (with the raw string and an error) instead of `tool_calls`. Always check both.
- `content_blocks` shows the same call as a standard `tool_call` block (Lesson 01.02), so provider-agnostic code can read it.

---
## Part 3 · `tool_choice`: may, must, or this one

| `tool_choice` | Meaning | `ChatGroq` sends |
|---|---|---|
| not set / `"auto"` | the model decides (default) | nothing / `"auto"` |
| `"any"` or `"required"` | must call *some* tool | `"required"` (`"any"` is translated) |
| `"calculator"` (a name) | must call *this* tool | `{"type": "function", "function": {"name": "calculator"}}` |
| `True` | must call the only tool (one tool only) | the same, with that tool's name |
| `"none"` | must not call tools | `"none"` |

Forcing has a cost: the model **must** call a tool even when none fits, so it will invent arguments. Use it when a tool call is the whole point (extraction, routing), not for chat. `create_agent` uses `"any"` internally for structured output (Lesson 02.05).

> **Groq limit:** the GPT-OSS models (`openai/gpt-oss-20b`/`-120b`) do **not** support *parallel* tool use ([tool use overview](https://console.groq.com/docs/tool-use/overview#supported-models)): they request one call per turn. A question that needs three lookups takes three round trips. Keep that in mind when you count latency and tokens.

In [ ]:
# 🧪 Offline: how ChatGroq translates each tool_choice value (inspect the binding, no call).
for choice in ("auto", "any", "calculator", "none"):
    print(f"{choice!r:<13} → {offline_groq.bind_tools([calculator, days_between], tool_choice=choice).kwargs['tool_choice']}")
print(f"{'True':<13} → {offline_groq.bind_tools([calculator], tool_choice=True).kwargs['tool_choice']}")
try:
    offline_groq.bind_tools([calculator, days_between], tool_choice=True)
except ValueError as exc:
    print("True with 2 tools →", exc)

---
## Part 4 · Executing tool calls and the message-ordering rules

To continue the conversation you append, in order:

1. the model's `AIMessage` (with its `tool_calls`), **unchanged**,
2. **one `ToolMessage` per tool call**, each with the matching `tool_call_id`,

and call the model again. Providers enforce this: an `AIMessage` with a tool call that has no answer, or a `ToolMessage` whose id matches nothing, is rejected with a 400 error ("tool call ids did not have response messages" or similar).

The easy way to get it right: `tool.invoke(tool_call)` with the **whole** call dict returns a `ToolMessage` with the right id.

In [ ]:
# 🧪 Offline: execute the scripted call and build the next request by hand.
TOOLS = {t.name: t for t in (calculator, days_between)}

history = [HumanMessage("How many days until Christmas 2026? Today is 2026-10-12."), reply]
for call in reply.tool_calls:
    tool_message = TOOLS[call["name"]].invoke(call)       # ToolCall in → ToolMessage out.
    history.append(tool_message)

for message in history:
    extra = f" tool_call_id={message.tool_call_id}" if isinstance(message, ToolMessage) else ""
    print(f"{type(message).__name__:<13}{extra} {message.content or message.tool_calls}")

**You should now see** `HumanMessage`, `AIMessage` (with the call), and `ToolMessage tool_call_id=call_abc123 74`. That history is a valid request: the model would now answer *"There are 74 days."*

---
## Part 5 · The loop, by hand

An agent is this loop:

```text
messages = [user question]
repeat at most N times:
    reply = model(messages)          # with tools bound
    messages += [reply]
    if reply has no tool calls: STOP → reply is the answer
    messages += [run each tool call]  # one ToolMessage per call
```

Two exits: the **stop condition** (no tool calls) and the **step limit** (protects you from loops and bills). Write it once yourself; from Lesson 02.03 on, `create_agent` runs it for you.

In [ ]:
# 🧪 Offline: the loop in ~15 lines, driven by a scripted model that needs TWO tools.
def tool_loop(model, tools, question: str, max_steps: int = 5) -> list:
    by_name = {t.name: t for t in tools}
    bound_model = model.bind_tools(tools)
    messages = [HumanMessage(question)]
    for step in range(1, max_steps + 1):
        ai = bound_model.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:                                   # Stop condition.
            print(f"done after {step} model calls")
            return messages
        for call in ai.tool_calls:
            messages.append(by_name[call["name"]].invoke(call))
    raise RuntimeError(f"no final answer after {max_steps} model calls")  # Step limit.


two_step = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "days_between", "args": {"start": "2026-10-12", "end": "2026-12-25"}, "id": "c1"}]),
    AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "74 * 45000"}, "id": "c2"}]),
    "There are 74 days left; saving 45,000 COP a day gives 3,330,000 COP by Christmas.",
])
messages = tool_loop(two_step, [calculator, days_between], "If I save 45,000 COP a day from 2026-10-12, how much do I have by 2026-12-25?")

from mastery.agents import format_transcript
print(format_transcript(messages))

In [ ]:
# 💸 Live: the same loop with the real model (3–4 short calls).
if HAVE_GROQ_KEY:
    live_messages = tool_loop(get_model("fast", temperature=0, reasoning_effort="low"), [calculator, days_between],
                              "If I save 45,000 COP a day from 2026-10-12 until 2026-12-25 (use days_between, "
                              "then the calculator), how much will I have? Answer in one sentence.")
    print(format_transcript(live_messages))
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

### What just happened?

- The model made **one tool call per turn** (no parallel calls on GPT-OSS), so the run needed three model calls: `days_between` → `calculator` → answer.
- **The model did no arithmetic.** It chose tools and arguments; Python computed `74` and `3330000`. That separation is the core idea of Project 02.
- `format_transcript` (from `mastery.agents`) prints one line per message. `mastery.agents.run_tool_loop` is the tested version of `tool_loop`, used later.

---
## Part 6 · When tool calls go wrong

Models make three kinds of mistakes: calling a tool that doesn't exist, sending arguments that don't validate, and sending valid arguments that fail inside the tool. **None of them should crash your program.** Each should become an error `ToolMessage` that tells the model what to fix. Our hand-written loop above would crash on all three. `mastery.agents.execute_tool_calls` handles them:

In [ ]:
# 🧪 Offline: three broken calls, three error ToolMessages, zero crashes.
from mastery.agents import execute_tool_calls

broken = AIMessage("", tool_calls=[
    {"name": "currency_converter", "args": {"amount": 10}, "id": "e1"},          # Unknown tool.
    {"name": "days_between", "args": {"start": "2026-10-12"}, "id": "e2"},        # Missing argument "end".
    {"name": "calculator", "args": {"expression": "10 / 0"}, "id": "e3"},         # Fails inside the tool.
])
for message in execute_tool_calls(broken, [calculator, days_between]):
    print(f"{message.tool_call_id} {message.status:<7} {' '.join(message.content.split())[:110]}")

In [ ]:
# 🧪 Offline: the model reads the error and fixes its call (scripted, so we can see the recovery).
from mastery.agents import run_tool_loop

recovering = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "17.5% of 2340"}, "id": "r1"}]),
    lambda msgs: AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "2340 * 0.175"}, "id": "r2"}])
    if msgs[-1].status == "error" else AIMessage("unexpected"),
    "17.5% of 2340 is 409.5.",
])
print(format_transcript(run_tool_loop(recovering, [calculator], [("user", "What is 17.5% of 2340?")])))

**You should now see** a ❌ tool line (*"Not a valid arithmetic expression"*), then a corrected call and the answer. Real models recover like this most of the time, **if** the error message says what to do. That's why the course's tools raise `ToolException` with a hint (*"Use ISO format YYYY-MM-DD"*), not a bare stack trace.

> **Don't leak internals in error messages.** The error text goes to the model provider and may reach the user. Name the problem and the fix; never include secrets, file paths or SQL (Lesson 02.04 adds `ToolErrorMiddleware` for this).

---
## Part 7 · Server-side tools: Groq's `browser_search`

Everything so far was **local** tool calling: the model asks, your Python runs. Groq also has **built-in** (server-side) tools: with `{"type": "browser_search"}` the GPT-OSS models search the web *on Groq's servers* and return a cited answer, all in **one** API call ([browser search](https://console.groq.com/docs/tool-use/built-in-tools/browser-search)). Lesson 05.01 covers it in depth; here's what you need to know now, verified in `langchain-groq` 1.1.3's source:

- `bind_tools([{"type": "browser_search"}])` **fails**: LangChain's converter only passes through tool types it knows (OpenAI's `web_search`, `code_interpreter`, ...), and `browser_search` isn't on that list.
- `model.bind(tools=[{"type": "browser_search"}])` **works**: `.bind()` attaches raw request parameters without conversion.
- The search steps come back in `additional_kwargs["executed_tools"]`.
- Not compatible with structured output in the same call; `reasoning_effort="low"` keeps token use down.

In [ ]:
# 🧪 Offline: bind_tools rejects the built-in tool; .bind() accepts it (no call is made).
try:
    offline_groq.bind_tools([{"type": "browser_search"}])
except ValueError as exc:
    print("bind_tools →", type(exc).__name__ + ":", str(exc).splitlines()[0])
searcher_offline = offline_groq.bind(tools=[{"type": "browser_search"}], tool_choice="required")
print("bind       →", searcher_offline.kwargs)

In [ ]:
# 💸 Live: one server-side search (uses more tokens than a normal call: page snippets count as input).
if HAVE_GROQ_KEY:
    searcher = get_model("fast", reasoning_effort="low").bind(tools=[{"type": "browser_search"}], tool_choice="required")
    found = searcher.invoke("In one sentence: which function replaced create_react_agent in LangChain v1?")
    print(found.text[-400:])
    print("\nexecuted_tools:", [t.get("type") for t in found.additional_kwargs.get("executed_tools", [])])
    print("tokens:", found.usage_metadata)
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** an answer mentioning `create_agent` with citation marks like `【1†L3-L5】`, and `executed_tools: ['browser_search', ...]`. Note there was **no** `tool_calls` field: the loop ran on Groq's side. You can't inspect or filter what the search read before the model used it, which is the main trade-off versus a client-side search tool like Tavily (Lesson 05.01).

---
## Part 8 · The OpenAI-compatible fallback

Groq's API is OpenAI-compatible, so `ChatOpenAI` from `langchain-openai` can talk to Groq by changing the base URL. Why would you? `langchain-groq` 1.1.3 still pins the Groq SDK below 1.0 (see `requirements.txt`), and a new Groq feature can reach the OpenAI client before the integration supports it. Having a tested fallback costs a few lines.

In [ ]:
# 💸 Live (optional package): the same tool call through ChatOpenAI pointed at Groq.
import importlib.util

if not importlib.util.find_spec("langchain_openai"):
    print("ℹ️ Optional: %pip install \"langchain-openai>=1.7.0,<2\", restart the kernel, and rerun this cell.")
elif HAVE_GROQ_KEY:
    from langchain_openai import ChatOpenAI

    groq_base = os.environ.get("GROQ_BASE_URL", "https://api.groq.com").rstrip("/") + "/openai/v1"
    compat = ChatOpenAI(model=settings.model_fast, base_url=groq_base, api_key=os.environ["GROQ_API_KEY"],
                        temperature=0, timeout=60, max_retries=2)
    compat_reply = compat.bind_tools([days_between]).invoke(QUESTION)
    print(type(compat).__name__, "→", compat_reply.tool_calls)
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** `ChatOpenAI → [{'name': 'days_between', ...}]`: same tool call, different client. What you lose: Groq-specific parsing (`reasoning_format`, `executed_tools`, `x_groq` usage details) and the Groq entry in `init_chat_model`. Use it as a fallback, not a default.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| 400: *tool call ids did not have response messages* | An `AIMessage` with `tool_calls` is not followed by a `ToolMessage` for every id | One `ToolMessage` per call, same `tool_call_id`, right after the `AIMessage` |
| The model "answers" with made-up numbers | It didn't call the tool (`tool_choice="auto"`) | Better description; tell it in the system prompt to always use tools for math; or force with `"required"` |
| `KeyError` on `call["name"]` in your loop | Model called a tool you didn't bind (or a typo) | Look up safely and return an error `ToolMessage` (`execute_tool_calls`) |
| Loop never ends | Model keeps calling tools | Step limit (`max_steps`); `create_agent` has `recursion_limit` |
| `ValueError: Unsupported function ... browser_search` | `bind_tools` can't convert Groq built-in tools | `model.bind(tools=[{"type": "browser_search"}])` |
| Arguments arrive as a string | You read `additional_kwargs["tool_calls"]` (raw) | Read `message.tool_calls` (parsed) and `invalid_tool_calls` |

## ⚠️ Production notes

> ⚠️ **Production note: every loop needs a budget.**
> A hand-written or framework loop must have a step limit, a timeout per model call (`get_model` sets 60 s), and ideally a token or cost budget per request. Log the number of steps per run: a sudden rise means a prompt, tool or model change broke something.

> ⚠️ **Production note: an OpenAI-compatible fallback for Groq.**
> `ChatOpenAI(base_url="https://api.groq.com/openai/v1", api_key=GROQ_API_KEY)` works for chat and local tool calling. Keep it tested (one smoke test in CI) so you can switch when `langchain-groq` lags behind the Groq API or its SDK pin blocks an upgrade. You lose Groq-specific parsing (reasoning, `executed_tools`, rate-limit details), so it's a fallback, not a default. Details: [docs/production-notes.md](../../docs/production-notes.md).

> ⚠️ **Production note: server-side tools move data and control to the provider.**
> With `browser_search`, Groq (and its search partner, Exa) see the query, and you can't filter pages before the model reads them. Groq states built-in tools are not available on regional/sovereign endpoints and are not HIPAA-covered. Check this before promising data residency.

## 🏋️ Exercises

1. **(Easy)** Rerun Part 2 live with `tool_choice="calculator"` for the date question. What arguments does the model invent?
2. **(Easy)** Count tokens: call `llm.invoke(QUESTION)` without tools and `llm_with_tools.invoke(QUESTION)` with them. How many input tokens do two tool schemas add?
3. **(Medium)** Replace the loop's `by_name[call["name"]].invoke(call)` with `execute_tool_calls`, then script a model that calls a tool that doesn't exist on step 1. Confirm the loop survives.
4. **(Stretch)** Make `tool_loop` print the **total** `input_tokens`/`output_tokens` over all steps (sum `usage_metadata`) and the estimated cost with `mastery.pricing.estimate_cost`. How does cost grow with the number of steps, and why (hint: every step resends the whole history)?

<details>
<summary><b>Show solutions</b></summary>

**2.** Typically 150–300 extra input tokens for two small tools; `days_between` and `calculator` have long, helpful descriptions. That's the price of good descriptions, paid on every step.

**4.**
```python
from mastery.pricing import estimate_cost
usage = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}
for m in live_messages:
    if isinstance(m, AIMessage) and m.usage_metadata:
        for k in usage:
            usage[k] += m.usage_metadata.get(k, 0)
print(usage, estimate_cost(usage, f"groq:{settings.model_fast}"))
```
Input tokens grow **quadratically-ish** with steps: step *n* resends the question, the tool schemas and all previous calls and results.
</details>

## ✅ Checkpoint

- [ ] I can show what `bind_tools` attaches to a request, and why tools cost tokens on every call.
- [ ] I can read `tool_calls`, `invalid_tool_calls` and `finish_reason`.
- [ ] I know what each `tool_choice` value does, and that GPT-OSS on Groq makes one tool call per turn.
- [ ] I can build a valid history: `AIMessage` with calls, then one `ToolMessage` per `tool_call_id`.
- [ ] I wrote the loop by hand, with a stop condition and a step limit, and it survives bad tool calls.
- [ ] I know why `bind_tools` rejects `browser_search` and how `.bind()` sends it anyway.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

In the `langchain-groq` source (`site-packages/langchain_groq/chat_models.py`, function `bind_tools`), find the line that drops `strict`. Why does it do that, and what does Groq's [structured outputs](https://console.groq.com/docs/structured-outputs) page say about strict mode for tools? Then search the [langchain GitHub issues](https://github.com/langchain-ai/langchain/issues?q=is%3Aissue%20groq%20browser_search) for `browser_search`: is there an open issue about `bind_tools` support? Log what you find in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `bind_tools` attaches tool schemas (and `tool_choice`) to every request; nothing runs until you call the model.
- A tool-calling reply has empty content and `tool_calls`; check `invalid_tool_calls` too.
- The loop: model → tools → model, until no tool calls or the step limit. One `ToolMessage` per call id.
- Errors become messages, not crashes, and good error text lets the model fix itself.
- Groq's `browser_search` is server-side and needs `.bind(...)`; `ChatOpenAI` with Groq's base URL is a tested fallback.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 02.03 · `create_agent`: the agent harness** → [`02-03-create-agent.ipynb`](02-03-create-agent.ipynb).

[Module 02 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)